# Asyncio timing — gather vs as_completed vs sequential

We use a fake slow function (asyncio.sleep) so the timing is dominated by IO.

In [ ]:
import asyncio, random, time

async def slow(i):
    await asyncio.sleep(random.uniform(0.05, 0.2))
    return i

async def sequential(n):
    return [await slow(i) for i in range(n)]

async def via_gather(n):
    return await asyncio.gather(*(slow(i) for i in range(n)))

async def via_as_completed(n):
    out = []
    for coro in asyncio.as_completed([slow(i) for i in range(n)]):
        out.append(await coro)
    return out

async def bench(label, fn, n):
    t = time.perf_counter()
    await fn(n)
    print(f'{label:20} n={n:3}  ms={(time.perf_counter()-t)*1000:7.1f}')

for n in (1, 5, 20, 50):
    await bench('sequential',    sequential,    n)
    await bench('gather',        via_gather,    n)
    await bench('as_completed',  via_as_completed, n)
    print('---')

## Reflect

- Why is `gather` ≈ `as_completed` in wall-clock time?
- When would you actually want `as_completed` (hint: streaming partial results)?